# Notebook 02 — Preprocessing Pipeline

## Purpose
Build and demonstrate the full preprocessing pipeline on our dataset.
Every step is shown with a before/after example.

The reusable implementation lives in `src/preprocessing.py`.
This notebook imports from there — it does not redefine any functions.

## Steps Covered
1. Text cleaning
2. Section extraction
3. Sentence segmentation
4. Stopword removal
5. Keyword extraction (KeyBERT + YAKE)
6. Named Entity Recognition (spaCy NER)

In [1]:
# ══════════════════════════════════════════════════════════════════════════════
#  GLOBAL CONFIG — match this to whatever you set in Notebook 01
# ══════════════════════════════════════════════════════════════════════════════
DATASET_MODE = "dev"   # "dev" | "full"
SEED = 42

import random, json, os, sys
random.seed(SEED)

# Make sure src/ is importable
sys.path.append(os.path.abspath(".."))

In [2]:
# Load the normalised splits saved by Notebook 01 — no re-downloading needed
data_dir = f"../data/processed_{DATASET_MODE}"

with open(f"{data_dir}/train.json", encoding="utf-8") as f:
    train_rows = json.load(f)

print(f"Loaded {len(train_rows)} train rows from {data_dir}/")
print(f"Fields in each row: {list(train_rows[0].keys())}")

Loaded 264 train rows from ../data/processed_dev/
Fields in each row: ['article', 'summary', 'domain', 'title', 'keywords']


In [3]:
from src.preprocessing import build_pipeline

# Load all NLP models once — reuse this object for all subsequent cells
pipeline = build_pipeline()

d:\H M Lahiru B H\Education\Faculty of Engineering\7 - Semester 7\EC 7203 (Advance AI)\Project\research-summarizer\.venv\Lib\site-packages\sentence_transformers\cross_encoder\CrossEncoder.py:13: TqdmExperimentalWarning: Using `tqdm.autonotebook.tqdm` in notebook mode. Use `tqdm.tqdm` instead to force console mode (e.g. in jupyter console)
  from tqdm.autonotebook import tqdm, trange


Loading spaCy model ...
Loading KeyBERT model ...


modules.json:   0%|          | 0.00/349 [00:01<?, ?B/s]

d:\H M Lahiru B H\Education\Faculty of Engineering\7 - Semester 7\EC 7203 (Advance AI)\Project\research-summarizer\.venv\Lib\site-packages\huggingface_hub\file_download.py:143: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in F:\huggingface\home\hub\models--sentence-transformers--all-MiniLM-L6-v2. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

...

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Loading YAKE extractor ...
Pipeline ready.


## 1. Text Cleaning

Removes LaTeX commands, citation markers, URLs, figure references,
and non-ASCII characters. Normalises whitespace.

In [4]:
from src.preprocessing import clean_text

sample = random.choice(train_rows)
raw_text = sample["article"]

cleaned = clean_text(raw_text)

print("── BEFORE (first 600 chars) ─────────────────────────────────────────")
print(raw_text[:600])
print("\n── AFTER cleaning (first 600 chars) ────────────────────────────────")
print(cleaned[:600])
print(f"\nLength before: {len(raw_text):,} chars")
print(f"Length after : {len(cleaned):,} chars")
print(f"Reduction    : {100*(1 - len(cleaned)/len(raw_text)):.1f}%")

── BEFORE (first 600 chars) ─────────────────────────────────────────
Currently, there are several widely used commercial cloud-based services that attempt to recognize an individual’s emotions
based on their facial expressions. Most research into facial emotion recognition has used high-resolution, front-oriented,
full-face images. However, when images are collected in naturalistic settings (e.g., using smartphone’s frontal camera), these
images are likely to be far from ideal due to camera positioning, lighting conditions, and camera shake. The impact these
conditions have on the accuracy of commercial emotion recognition services has not been studied in f

── AFTER cleaning (first 600 chars) ────────────────────────────────
Currently, there are several widely used commercial cloud-based services that attempt to recognize an individuals emotions based on their facial expressions. Most research into facial emotion recognition has used high-resolution, front-oriented, full-face images.

## 2. Section Extraction

Uses regex heuristics to identify and extract named sections:
Abstract, Introduction, Methodology, Results, Discussion, Conclusion.

In the dev dataset these sections may not always be explicitly labelled
(since articles are abstract+conclusion only). On the full arXiv/PubMed
datasets, section headers are much more common.

In [5]:
from src.preprocessing import extract_sections

sections = extract_sections(cleaned)

print("── Extracted Sections ───────────────────────────────────────────────")
for name, content in sections.items():
    status = f"{len(content)} chars" if content.strip() else "NOT FOUND"
    preview = content[:200].strip() if content.strip() else ""
    print(f"\n[{name.upper()}]  ({status})")
    if preview:
        print(preview + ("..." if len(content) > 200 else ""))

── Extracted Sections ───────────────────────────────────────────────

[ABSTRACT]  (NOT FOUND)

[INTRODUCTION]  (2536 chars)
Currently, there are several widely used commercial cloud-based services that attempt to recognize an individuals emotions based on their facial expressions. Most research into facial emotion recognit...

[METHODOLOGY]  (NOT FOUND)

[RESULTS]  (NOT FOUND)

[DISCUSSION]  (NOT FOUND)

[CONCLUSION]  (NOT FOUND)


## 3. Sentence Segmentation

Uses spaCy's dependency parser to detect sentence boundaries.
More accurate than simple period-splitting — handles abbreviations,
decimal numbers, and multi-sentence paragraphs correctly.

In [6]:
from src.preprocessing import segment_sentences

sentences = segment_sentences(cleaned, pipeline)

print(f"Total sentences detected: {len(sentences)}\n")
print("── First 8 sentences ───────────────────────────────────────────────")
for i, sent in enumerate(sentences[:8], 1):
    print(f"  [{i}] {sent}")

Total sentences detected: 14

── First 8 sentences ───────────────────────────────────────────────
  [1] Currently, there are several widely used commercial cloud-based services that attempt to recognize an individuals emotions based on their facial expressions.
  [2] Most research into facial emotion recognition has used high-resolution, front-oriented, full-face images.
  [3] However, when images are collected in naturalistic settings (e.g., using smartphones frontal camera), these images are likely to be far from ideal due to camera positioning, lighting conditions, and camera shake.
  [4] The impact these conditions have on the accuracy of commercial emotion recognition services has not been studied in full detail.
  [5] To fill this gap, we selected five prominent commercial emotion recognition systemsAmazon Rekognition, Baidu Research, Face++, Microsoft Azure, and Affectivaand evaluated their performance via two experiments.
  [6] In Experiment 1, we compared the systems accuracy

## 4. Stopword Removal

Removes common English stopwords (the, is, at, which, etc.) using NLTK.
The output is used downstream for keyword extraction — not for model input,
since transformer models handle stopwords internally.

In [7]:
from src.preprocessing import remove_stopwords

no_sw = remove_stopwords(cleaned, pipeline)

print("── BEFORE (first 300 chars) ─────────────────────────────────────────")
print(cleaned[:300])
print("\n── AFTER stopword removal (first 300 chars) ─────────────────────────")
print(no_sw[:300])
print(f"\nToken count before: {len(cleaned.split()):,}")
print(f"Token count after : {len(no_sw.split()):,}")
print(f"Stopwords removed : {len(cleaned.split()) - len(no_sw.split()):,}")

── BEFORE (first 300 chars) ─────────────────────────────────────────
Currently, there are several widely used commercial cloud-based services that attempt to recognize an individuals emotions based on their facial expressions. Most research into facial emotion recognition has used high-resolution, front-oriented, full-face images. However, when images are collected i

── AFTER stopword removal (first 300 chars) ─────────────────────────
currently, several widely used commercial cloud-based services attempt recognize individuals emotions based facial expressions. research facial emotion recognition used high-resolution, front-oriented, full-face images. however, images collected naturalistic settings (e.g., using smartphones frontal

Token count before: 354
Token count after : 233
Stopwords removed : 121


## 5. Keyword Extraction

We use two complementary methods:

- **KeyBERT** — embedding-based. Finds phrases semantically similar to
  the document. Captures domain-specific terminology well.
- **YAKE** — statistical, unsupervised. Uses word position, frequency,
  and co-occurrence. Fast and requires no GPU.

We combine both into a deduplicated list for the best coverage.
"""

In [8]:
from src.preprocessing import (
    extract_keywords_keybert,
    extract_keywords_yake,
    extract_keywords_combined,
)

kb_kws   = extract_keywords_keybert(cleaned, pipeline, top_n=10)
yake_kws = extract_keywords_yake(cleaned, pipeline, top_n=10)
combined = extract_keywords_combined(cleaned, pipeline, top_n=10)

print("── KeyBERT keywords (keyword, relevance score) ──────────────────────")
for kw, score in kb_kws:
    print(f"  {kw:<35}  {score:.4f}")

print("\n── YAKE keywords (keyword, score — lower = more relevant) ───────────")
for kw, score in yake_kws:
    print(f"  {kw:<35}  {score:.6f}")

print("\n── Combined (deduplicated) ───────────────────────────────────────────")
for kw in combined:
    print(f"  {kw}")

── KeyBERT keywords (keyword, relevance score) ──────────────────────
  face images                          0.5298
  emotion recognition                  0.5105
  facial emotion                       0.4942
  facial expressions                   0.4761
  face microsoft                       0.4704
  azure affectivaand                   0.4485
  facial expression                    0.4463
  emotions based                       0.4416
  azure affectiva                      0.4346
  individuals emotions                 0.4306

── YAKE keywords (keyword, score — lower = more relevant) ───────────
  emotion recognition                  0.023182
  emotion                              0.043356
  facial emotion                       0.044007
  recognition                          0.056380
  Microsoft Azure                      0.058884
  facial                               0.064998
  images                               0.084330
  Face                                 0.092849
  individuals e

## 6. Named Entity Recognition

spaCy's NER identifies real-world entities: organisations, people,
locations, dates, products. For medical papers this surfaces drug names,
diseases, and clinical terms (labelled as ORG or PRODUCT by en_core_web_sm).
A domain-specific model like en_core_sci_sm would do better on full medical
text — noted as future work.

In [9]:
from src.preprocessing import extract_entities

entities = extract_entities(cleaned, pipeline)

print("── Named Entities ───────────────────────────────────────────────────")
if entities:
    for label, ents in sorted(entities.items()):
        print(f"\n  [{label}]")
        for e in ents[:8]:   # show up to 8 per type
            print(f"    • {e}")
else:
    print("  No entities found in this sample.")

── Named Entities ───────────────────────────────────────────────────

  [CARDINAL]
    • five
    • two
    • three
    • 2
    • 1

  [GPE]
    • Affectivaand

  [ORDINAL]
    • first

  [ORG]
    • Baidu Research
    • Microsoft Azure
    • Amazon
    • Baidu
    • Affectiva


## 7. Full Pipeline — Single Entry Point

`preprocess()` runs all six steps in order and returns a single dict.
This is what every other notebook calls.

In [11]:
from src.preprocessing import preprocess

result = preprocess(sample["article"], pipeline)

print("── Full pipeline output keys ─────────────────────────────────────────")
for key, val in result.items():
    if isinstance(val, str):
        print(f"  {key:<15}: {len(val):>6,} chars   preview: {val[:80].strip()!r}")
    elif isinstance(val, list):
        print(f"  {key:<15}: {len(val):>6,} items")
    elif isinstance(val, dict):
        if key == "entities":
            # entities dict has list values, not strings
            found = [f"{label}({len(ents)})" for label, ents in val.items()]
            print(f"  {key:<15}: {len(val):>6,} types    found: {found}")
        else:
            # sections dict has string values
            found = [k for k, v in val.items() if isinstance(v, str) and v.strip()]
            print(f"  {key:<15}: {len(val):>6,} sections  found: {found}")

── Full pipeline output keys ─────────────────────────────────────────
  cleaned_text   :  2,536 chars   preview: 'Currently, there are several widely used commercial cloud-based services that at'
  sections       :      6 sections  found: ['introduction']
  sentences      :     14 items
  no_stopwords   :  2,059 chars   preview: 'currently, several widely used commercial cloud-based services attempt recognize'
  keywords       :     10 items
  entities       :      4 types    found: ['CARDINAL(5)', 'ORG(5)', 'GPE(1)', 'ORDINAL(1)']


In [12]:
import json

print(f"Running full pipeline on {len(train_rows)} train rows ...")
print("(This may take 1–2 minutes in dev mode)\n")

preprocessed = []
for i, row in enumerate(train_rows):
    result = preprocess(row["article"], pipeline)
    preprocessed.append({
        "domain"       : row["domain"],
        "title"        : row.get("title", ""),
        "summary"      : row["summary"],
        "keywords"     : row.get("keywords", ""),
        **result,       # cleaned_text, sections, sentences, no_stopwords, keywords, entities
    })
    if (i + 1) % 50 == 0:
        print(f"  Processed {i+1}/{len(train_rows)}")

# Save
out_dir = f"../data/preprocessed_{DATASET_MODE}"
os.makedirs(out_dir, exist_ok=True)
out_path = f"{out_dir}/train_preprocessed.json"

with open(out_path, "w", encoding="utf-8") as f:
    json.dump(preprocessed, f, ensure_ascii=False, indent=2)

print(f"\nSaved {len(preprocessed)} preprocessed rows → {out_path}")

Running full pipeline on 264 train rows ...
(This may take 1–2 minutes in dev mode)

  Processed 50/264
  Processed 100/264
  Processed 150/264
  Processed 200/264
  Processed 250/264

Saved 264 preprocessed rows → ../data/preprocessed_dev/train_preprocessed.json


## ✅ Phase 2 Complete — Gate Check

- [ ] `src/preprocessing.py` created with all functions and docstrings
- [ ] Text cleaning removes LaTeX, URLs, citations correctly
- [ ] Section extraction finds at least some sections (more visible on full data)
- [ ] Sentence segmentation produces sensible sentence boundaries
- [ ] Stopword removal reduces token count meaningfully
- [ ] KeyBERT + YAKE both return domain-relevant keywords
- [ ] NER finds entities in the sample text
- [ ] Full pipeline runs on all train rows without errors
- [ ] Preprocessed data saved to `data/preprocessed_dev/`

Proceed to `03_domain_classifier.ipynb`.